# Il primo modello



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Selezionare i dati per il modello
Il dataset conteneva troppe variabili per riuscire a comprenderle tutte, o anche solo per stamparle in modo leggibile. Come puoi ridurre questa quantità di dati a qualcosa di comprensibile?

Cominceremo scegliendo poche variabili sulla base del nostro intuito. Nei corsi successivi vedrai tecniche statistiche per stabilire automaticamente quali variabili privilegiare.

Per scegliere le variabili, cioè le colonne, dobbiamo vedere un elenco di tutte le colonne del dataset. Lo otteniamo con la proprietà **columns** del DataFrame, usata nell’ultima riga del codice seguente.


In [ ]:
import pandas as pd

melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
melbourne_data.columns

In [ ]:
# Nei dati di Melbourne mancano alcuni valori: per certe case non sono state registrate tutte le variabili.
# Vedremo come gestire i valori mancanti in una lezione successiva.
# I dati Iowa non hanno valori mancanti nelle colonne utilizzate.
# Per ora scegliamo la soluzione più semplice: escludiamo le case con dati mancanti.
# Per ora non è necessario approfondire; il codice è:

# dropna elimina i valori mancanti; na significa "non disponibile".
melbourne_data = melbourne_data.dropna(axis=0)

Ci sono molti modi per selezionare un sottoinsieme dei dati. Il [corso pandas](https://www.kaggle.com/learn/pandas) li tratta più approfonditamente, ma per ora ci concentreremo su due approcci:

1. La notazione con il punto, che useremo per selezionare il «target della previsione».
2. La selezione tramite un elenco di colonne, che useremo per scegliere le «feature».

## Selezionare il target della previsione
Puoi estrarre una variabile con la **notazione con il punto**. Questa singola colonna viene memorizzata in una **Series**, che in termini generali è simile a un DataFrame con una sola colonna di dati.

Useremo la notazione con il punto per selezionare la colonna da prevedere, chiamata **target della previsione**. Per convenzione, il target viene chiamato **y**. Il codice necessario per salvare i prezzi delle case nei dati di Melbourne è quindi:


In [ ]:
y = melbourne_data.Price

# Scegliere le «feature»
Le colonne fornite in ingresso al modello, e usate in seguito per fare previsioni, si chiamano **feature**, cioè caratteristiche. Nel nostro caso sono le colonne utilizzate per determinare il prezzo di una casa. A volte userai come feature tutte le colonne tranne il target; altre volte sarà meglio usarne meno.

Per ora costruiremo un modello con poche feature. Più avanti vedrai come procedere per tentativi e confrontare modelli costruiti con feature diverse.

Selezioniamo più feature passando un elenco di nomi di colonne tra parentesi quadre. Ogni elemento dell’elenco deve essere una stringa, quindi va racchiuso tra virgolette.

Ecco un esempio:


In [ ]:
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']

Per convenzione, questi dati vengono chiamati **X**.


In [ ]:
X = melbourne_data[melbourne_features]

Esaminiamo rapidamente i dati che useremo per prevedere i prezzi delle case, con il metodo `describe` e il metodo `head`, che mostra le prime righe.


In [ ]:
X.describe()

In [ ]:
X.head()

Controllare visivamente i dati con questi comandi è una parte importante del lavoro di un data scientist. Troverai spesso sorprese nel dataset che meritano ulteriori approfondimenti.


---
# Costruire il modello

Userai la libreria **scikit-learn** per creare i modelli. Nel codice il nome della libreria si scrive **sklearn**, come vedrai nell’esempio. Scikit-learn è di gran lunga la libreria più diffusa per modellare i tipi di dati normalmente memorizzati nei DataFrame.

I passaggi per costruire e usare un modello sono:

* **Definire:** quale tipo di modello sarà? Un albero decisionale? Un altro tipo? In questa fase si specificano anche alcuni parametri del tipo di modello scelto.
* **Addestrare, con fit:** ricavare schemi dai dati forniti. È il cuore della modellazione.
* **Prevedere, con predict:** esattamente ciò che suggerisce il nome.
* **Valutare:** determinare quanto siano accurate le previsioni del modello.

Ecco un esempio di definizione di un albero decisionale con scikit-learn e di addestramento sulle feature e sulla variabile target.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Definisci il modello; assegna un numero a random_state per ottenere risultati riproducibili.
melbourne_model = DecisionTreeRegressor(random_state=1)

# Addestra il modello.
melbourne_model.fit(X, y)

Molti modelli di machine learning introducono una componente casuale durante l’addestramento. Specificare un numero per `random_state` garantisce gli stessi risultati a ogni esecuzione ed è considerato una buona pratica. Puoi usare qualsiasi numero: la qualità del modello non dipenderà in modo significativo dal valore preciso scelto.

Ora abbiamo un modello addestrato che possiamo usare per fare previsioni.

Nella pratica vorrai fare previsioni per nuove case immesse sul mercato, anziché per quelle di cui conosciamo già il prezzo. Ma, per capire come funziona `predict`, faremo previsioni sulle prime righe dei dati di training.


In [ ]:
print("Making predictions for the following 5 houses:")
print(X.head())
print("The predictions are")
print(melbourne_model.predict(X.head()))

# Ora tocca a te
Prova tu nell’**[esercizio sulla costruzione di un modello](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex3.html)**.
